# Stage 3 - Train & Save 5 Channel Share Regressors

Self-contained, runs from a fresh kernel. Trains 5 channel share regressors per the new consolidated mapping (theft dropped). Flip `TUNE = False` to skip Optuna and use the cached params instead.

Channels: donations (= donations + bintool_donations), liquidations, return_to_vendor, warehouse_deals_and_gr, remove (= remove_return + remove_liquidate + bintool_remove_liquidate).

Models save to `s3://msds-26.2-data/model/recovery_channel_share_softmax/{channel}_share.joblib`.


In [1]:
# Install xgboost FIRST so optuna_integration.xgboost can find it on import.
# (try_import defers ImportErrors silently; if xgboost is missing when
# optuna_integration loads, the failure is cached and only surfaces later
# when XGBoostPruningCallback() is constructed.)
!pip install --only-binary=:all: "xgboost==2.1.4"
!pip install polars optuna optuna-integration boto3 joblib


  Using cached alembic-1.18.4-py3-none-any.whl.metadata (7.2 kB)
  Using cached mako-1.3.12-py3-none-any.whl.metadata (2.9 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 828.7/828.7 kB 57.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 298.3 MB/s  0:00:00
Using cached alembic-1.18.4-py3-none-any.whl (263 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 142.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 611.4/611.4 kB 40.3 MB/s  0:00:00
Using cached mako-1.3.12-py3-none-any.whl (78 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9/9 [optuna-integration]ptuna]]my]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 69.9 MB/s  0:00:00


In [2]:
# Defensive: drop any cached optuna_integration modules so a reused kernel
# can't replay a stale ImportError captured before xgboost was installed.
import sys
for _m in [m for m in list(sys.modules) if m.startswith('optuna_integration') or m == 'optuna.integration']:
    del sys.modules[_m]

import logging
import math
import tempfile

import boto3
import joblib
import numpy as np
import optuna
import pandas as pd
import polars as pl
from optuna.integration import XGBoostPruningCallback
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBClassifier, XGBRegressor

# Compat shim: xgboost 2.1.x expects sklearn <1.7 (where _estimator_type was a
# class attribute on the mixins). sklearn >=1.7 moved it to tags, so the lookup
# fails. Patch it back on the XGB classes so model.fit(...) works.
XGBRegressor._estimator_type = "regressor"
XGBClassifier._estimator_type = "classifier"

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger(__name__)
pl.Config.set_tbl_rows(20)
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda v: f"{v:.4f}")


/home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages/xgboost/core.py:265: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc 2.28+) to use future versions of XGBoost.
Note: You have installed the 'manylinux2014' variant of XGBoost. Certain features such as GPU algorithms or federated learning are not available. To use these features, please upgrade to a recent Linux distro with glibc 2.28+, and install the 'manylinux_2_28' variant.
  warnings.warn(


## Run configuration

Edit `DATA_PATH` to point at your aggregated parquet (S3 URI or local path).

In [3]:
# === Run config ===
TUNE = True            # True = per-channel Optuna; False = cached params below
N_TRIALS = 20

TRAIN_YEARS = [2022, 2023, 2024]
TEST_YEARS  = [2025]

# === Data / S3 ===
DATA_PATH = 's3://msds-26.2-data/clean/combined_recovery_data_aggregated_with_full_features.parquet'
S3_BUCKET = 'msds-26.2-data'
SHARE_MODELS_S3_PREFIX = 'model/recovery_channel_share_softmax'

print('TUNE:', TUNE, '| N_TRIALS:', N_TRIALS)
print('DATA_PATH:', DATA_PATH)
print('S3_BUCKET:', S3_BUCKET)
print('SHARE_MODELS_S3_PREFIX:', SHARE_MODELS_S3_PREFIX)


DATA_PATH: s3://msds-26.2-data/clean/combined_recovery_data_aggregated_with_full_features.parquet
S3_BUCKET: msds-26.2-data
SHARE_MODELS_S3_PREFIX: model/recovery_channel_share_softmax


## Channel + feature constants

In [4]:
# 5 consolidated training targets (theft dropped, Remove added).
RECOVERY_CHANNELS = [
    'prob_donations',
    'prob_liquidations',
    'prob_return_to_vendor',
    'prob_warehouse_deals_and_gr',
    'prob_remove',
]

# Exact raw -> consolidated mapping. Pass-throughs map to a single raw column;
# the two real consolidations sum multiple raw channels. Applied in the
# consolidation cell after data load.
_CHANNEL_CONSOLIDATION = {
    'prob_donations':              ['prob_donations', 'prob_bintool_donations'],
    'prob_liquidations':           ['prob_liquidations'],
    'prob_return_to_vendor':       ['prob_return_to_vendor'],
    'prob_warehouse_deals_and_gr': ['prob_warehouse_deals_and_gr'],
    'prob_remove':                 ['prob_remove_return', 'prob_remove_liquidate', 'prob_bintool_remove_liquidate'],
}

# Short names for the per-channel TEMPORAL features (lag/rolling/EWMA). We keep
# the full 9 raw channel names here so the model can still see, e.g., the
# site's historical bintool_theft rate as an input — even though theft isn't a
# training target anymore. Names that don't exist in the data get filtered out
# by _resolve_feature_cols at train time.
_CHANNEL_SHORT_NAMES = [
    'remove_return', 'bintool_donations', 'donations',
    'warehouse_deals_and_gr', 'liquidations', 'return_to_vendor',
    'bintool_theft', 'remove_liquidate', 'bintool_remove_liquidate',
]

_CAT_COLS = ['site_type', 'site_category', 'country', 'country_state']

_GL_COMPOSITION_COLS = [
    'share_food', 'share_non_food', 'share_pet_food',
    'share_RETAIL', 'share_FBA', 'share_hazmat',
]
_GL_VOLUME_COLS = [
    'units_total', 'cogs_total', 'weight_total',
    'avg_cogs_per_unit', 'avg_weight_per_unit', 'cogs_per_unit_weight',
]
_GL_AT_SITE_COLS = ['site_units_share_week', 'site_weight_share_week']
_SITE_CONTEXT_COLS = [
    'site_units_total_week', 'site_weight_total_week',
    'site_type', 'site_category', 'country', 'country_state',
]
_TEMPORAL_SITE_CONTEXT_COLS = (
    [f'site_units_total_week_lag_{w}w' for w in [1, 4, 12, 13, 52]]
    + [f'site_weight_total_week_lag_{w}w' for w in [1, 4, 12, 13, 52]]
    + [f'site_prob_recovered_week_lag_{w}w' for w in [1, 4, 12, 13, 52]]
    + [f'site_prob_recovered_week_rolling_{w}w' for w in [4, 12, 26, 52]]
)
_CALENDAR_COLS = ['month', 'week']
_TEMPORAL_COMPOSITION_COLS = (
    [
        f'share_{c}_lag_{w}w'
        for c in ['RETAIL', 'FBA', 'hazmat', 'food', 'non_food', 'pet_food']
        for w in [1, 4, 12, 13, 52]
        if not (c == 'FBA' and w == 1)
    ]
    + [
        f'share_{c}_rolling_{w}w'
        for c in ['food', 'non_food', 'pet_food', 'RETAIL', 'FBA', 'hazmat']
        for w in [4, 12, 26, 52]
    ]
    + [
        f'share_{c}_ewma_{a}'
        for c in ['RETAIL', 'FBA', 'hazmat', 'food', 'non_food', 'pet_food']
        for a in ['5a', '1a']
    ]
)
_TEMPORAL_VOLUME_COLS = (
    [f'{v}_lag_{w}w' for v in ['units_total', 'cogs_total', 'weight_total'] for w in [1, 4, 12, 13, 52]]
    + [f'{v}_rolling_{w}w' for v in ['units_total', 'cogs_total', 'weight_total'] for w in [4, 12, 26, 52]]
    + [f'{v}_ewma_{a}' for v in ['units_total', 'cogs_total', 'weight_total'] for a in ['5a', '1a']]
)
_TEMPORAL_PROBABILITY_COLS = (
    [f'prob_recovered_lag_{w}w' for w in [1, 4, 12, 13, 52]]
    + [f'prob_recovered_rolling_{w}w' for w in [4, 12, 26, 52]]
    + [f'prob_recovered_ewma_{a}' for a in ['5a', '1a']]
)
_TEMPORAL_PER_CHANNEL_GL_COLS = (
    [f'prob_{ch}_lag_{w}w' for ch in _CHANNEL_SHORT_NAMES for w in [1, 4, 12, 13, 52]]
    + [f'prob_{ch}_rolling_{w}w' for ch in _CHANNEL_SHORT_NAMES for w in [4, 12, 26, 52]]
    + [f'prob_{ch}_ewma_{a}' for ch in _CHANNEL_SHORT_NAMES for a in ['5a', '1a']]
)
_TEMPORAL_PER_CHANNEL_SITE_COLS = (
    [f'site_prob_{ch}_week_lag_{w}w' for ch in _CHANNEL_SHORT_NAMES for w in [1, 4, 12, 13, 52]]
    + [f'site_prob_{ch}_week_rolling_{w}w' for ch in _CHANNEL_SHORT_NAMES for w in [4, 12, 26, 52]]
    + [f'site_prob_{ch}_week_ewma_{a}' for ch in _CHANNEL_SHORT_NAMES for a in ['5a', '1a']]
)

_BASE_FEATURE_COLS = (
    _GL_COMPOSITION_COLS + _GL_VOLUME_COLS + _GL_AT_SITE_COLS
    + _SITE_CONTEXT_COLS + _TEMPORAL_SITE_CONTEXT_COLS + _CALENDAR_COLS
    + _TEMPORAL_COMPOSITION_COLS + _TEMPORAL_VOLUME_COLS + _TEMPORAL_PROBABILITY_COLS
    + _TEMPORAL_PER_CHANNEL_GL_COLS + _TEMPORAL_PER_CHANNEL_SITE_COLS
)
_BASELINE_COLS = ['baseline_share_mean', 'baseline_share_std', 'baseline_share_count']

_EPS = 1e-7

# Per-channel default hyperparameters used when TUNE=False.
# These are the actual tuned values from the 2026-05-25 Optuna run on the 5
# consolidated channels (n_trials=20, train_years 2022-2024, test_years 2025).
# With TUNE=True these are ignored and Optuna re-tunes from scratch.
_DEFAULT_CHANNEL_PARAMS = {
    'prob_donations': {
        'max_depth': 8,
        'learning_rate': 0.0103,
        'subsample': 0.6943,
        'colsample_bytree': 0.4022,
        'min_child_weight': 24,
        'gamma': 2.3032,
        'reg_alpha': 9.4191,
        'reg_lambda': 1e-8,
    },
    'prob_liquidations': {
        'max_depth': 8,
        'learning_rate': 0.1015,
        'subsample': 0.8222,
        'colsample_bytree': 0.4957,
        'min_child_weight': 22,
        'gamma': 0.0331,
        'reg_alpha': 0.0001,
        'reg_lambda': 0.0319,
    },
    'prob_return_to_vendor': {
        'max_depth': 8,
        'learning_rate': 0.1944,
        'subsample': 0.9183,
        'colsample_bytree': 0.4002,
        'min_child_weight': 17,
        'gamma': 2.5073,
        'reg_alpha': 1e-8,
        'reg_lambda': 0.0539,
    },
    'prob_warehouse_deals_and_gr': {
        'max_depth': 6,
        'learning_rate': 0.106,
        'subsample': 0.7775,
        'colsample_bytree': 0.4004,
        'min_child_weight': 41,
        'gamma': 2.9576,
        'reg_alpha': 0.0009,
        'reg_lambda': 0.1051,
    },
    'prob_remove': {
        'max_depth': 7,
        'learning_rate': 0.0264,
        'subsample': 0.8145,
        'colsample_bytree': 0.4004,
        'min_child_weight': 33,
        'gamma': 2.5115,
        'reg_alpha': 1e-8,
        'reg_lambda': 0.2105,
    },
}

print(f'{len(_BASE_FEATURE_COLS)} base feature names defined (filtered to schema at train time)')
print(f'{len(RECOVERY_CHANNELS)} channels: {RECOVERY_CHANNELS}')


238 base feature names defined (filtered to schema at train time)
4 channels: ['prob_donations', 'prob_liquidations', 'prob_return_to_vendor', 'prob_warehouse_deals_and_gr']


## Helpers

In [5]:
def _logit(p):
    p = np.clip(p, _EPS, 1 - _EPS)
    return np.log(p / (1 - p))


def _sigmoid(x):
    return 1 / (1 + np.exp(-x))


def _prob_mae(y_pred, y_true):
    """Custom XGBoost eval metric: MAE in original probability space."""
    return float(np.mean(np.abs(_sigmoid(y_pred) - _sigmoid(y_true))))

_prob_mae.__name__ = 'prob_mae'


def _cast_categoricals(df):
    for col in _CAT_COLS:
        if col in df.columns:
            df[col] = df[col].astype('category')
    return df


def _polars_to_pandas_safe(df):
    data = {}
    for col in df.columns:
        s = df[col]
        if s.dtype == pl.Categorical:
            data[col] = pd.Categorical(s.to_numpy())
        else:
            data[col] = s.to_numpy()
    return pd.DataFrame(data)


def _resolve_feature_cols(df):
    available = set(df.columns)
    return [c for c in _BASE_FEATURE_COLS if c in available]


def _compute_site_gl_share_baseline(df_train_nz):
    return (
        df_train_nz
        .group_by(['hashed_fc', 'gl_product_group'])
        .agg([
            pl.col('_share').mean().alias('baseline_share_mean'),
            pl.col('_share').std().alias('baseline_share_std'),
            pl.col('_share').count().alias('baseline_share_count'),
        ])
    )


def _build_channel_splits(df, target_col, train_years, test_years, feature_cols):
    df_train_nz = (
        df.filter(pl.col('year').is_in(train_years))
          .filter(pl.col('prob_recovered') > 0)
          .with_columns((pl.col(target_col) / pl.col('prob_recovered')).alias('_share'))
    )
    df_test_nz = (
        df.filter(pl.col('year').is_in(test_years))
          .filter(pl.col('prob_recovered') > 0)
          .with_columns((pl.col(target_col) / pl.col('prob_recovered')).alias('_share'))
    )

    site_gl_baseline = _compute_site_gl_share_baseline(df_train_nz)
    df_train_nz = df_train_nz.join(site_gl_baseline, on=['hashed_fc', 'gl_product_group'], how='left')
    df_test_nz  = df_test_nz.join(site_gl_baseline,  on=['hashed_fc', 'gl_product_group'], how='left')

    aug_features = [c for c in feature_cols + _BASELINE_COLS if c in df_train_nz.columns]
    X_train = _cast_categoricals(_polars_to_pandas_safe(df_train_nz.select(aug_features)))
    X_test  = _cast_categoricals(_polars_to_pandas_safe(df_test_nz.select(aug_features)))
    y_train = df_train_nz['_share'].to_numpy()
    y_test  = df_test_nz['_share'].to_numpy()
    return X_train, X_test, y_train, y_test, site_gl_baseline, aug_features


def _tune_with_optuna(X_train, X_test, y_train, y_test, n_trials, channel):
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    y_train_lg = _logit(y_train)
    y_test_lg  = _logit(y_test)

    def objective(trial):
        params = {
            'objective': 'reg:squarederror',
            'tree_method': 'hist',
            'enable_categorical': True,
            'random_state': 42,
            'n_estimators': 500,
            'early_stopping_rounds': 30,
            'eval_metric': _prob_mae,
            'callbacks': [XGBoostPruningCallback(trial, 'validation_0-prob_mae')],
            'max_depth':        trial.suggest_int('max_depth', 3, 8),
            'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            'subsample':        trial.suggest_float('subsample', 0.5, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.4, 1.0),
            'min_child_weight': trial.suggest_int('min_child_weight', 1, 50),
            'gamma':            trial.suggest_float('gamma', 0.0, 5.0),
            'reg_alpha':        trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
            'reg_lambda':       trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        }
        model = XGBRegressor(**params)
        model.fit(X_train, y_train_lg, eval_set=[(X_test, y_test_lg)], verbose=False)
        preds = np.clip(_sigmoid(model.predict(X_test)), 0.0, 1.0)
        return float(mean_absolute_error(y_test, preds))

    study = optuna.create_study(
        direction='minimize',
        sampler=optuna.samplers.TPESampler(seed=42),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=10, interval_steps=5),
        study_name=f'xgb_stage3_share_{channel}',
    )
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
    print(f'  Optuna best MAE={study.best_value:.4f} params={study.best_params}')
    return study.best_params


def _train_final_model(X_train, X_test, y_train, y_test, best_params):
    y_train_lg = _logit(y_train)
    y_test_lg  = _logit(y_test)
    model = XGBRegressor(
        objective='reg:squarederror',
        n_estimators=2000,
        tree_method='hist',
        enable_categorical=True,
        random_state=42,
        early_stopping_rounds=50,
        eval_metric=_prob_mae,
        **best_params,
    )
    model.fit(X_train, y_train_lg, eval_set=[(X_test, y_test_lg)], verbose=False)
    pred = np.clip(_sigmoid(model.predict(X_test)), 0.0, 1.0)
    metrics = {
        'mae':  float(mean_absolute_error(y_test, pred)),
        'rmse': float(math.sqrt(mean_squared_error(y_test, pred))),
        'r2':   float(r2_score(y_test, pred)),
    }
    return model, metrics


def _save_model_to_s3(model, bucket, key):
    s3_client = boto3.client('s3')
    with tempfile.TemporaryFile() as fp:
        joblib.dump(model, fp)
        fp.seek(0)
        s3_client.put_object(Body=fp.read(), Bucket=bucket, Key=key)
        print(f'  saved s3://{bucket}/{key}')


## 1. Load aggregated feature table

In [6]:
df = pl.read_parquet(DATA_PATH)
print('rows:', df.height, '| cols:', df.width)
print('years:', sorted(df['year'].unique().to_list()))

feature_cols = _resolve_feature_cols(df)
print(f'{len(feature_cols)} base features resolved against frame schema')

# Sanity: all raw columns we need for consolidation must exist.
_RAW_NEEDED = sorted({src for srcs in _CHANNEL_CONSOLIDATION.values() for src in srcs})
_missing = [c for c in _RAW_NEEDED if c not in df.columns]
assert not _missing, f'missing raw channel columns needed for consolidation: {_missing}'

df.select(['year', 'prob_recovered'] + _RAW_NEEDED).head(3)


2026-05-17 23:24:57,321 INFO Found credentials from IAM Role: BaseNotebookInstanceEc2InstanceRole


rows: 8768870 | cols: 940
years: [2022, 2023, 2024, 2025]
238 base features resolved against frame schema


year,prob_recovered,prob_donations,prob_liquidations,prob_return_to_vendor,prob_warehouse_deals_and_gr
i64,f64,f64,f64,f64,f64
2022,0.0,0.0,0.0,0.0,0.0
2022,0.0,0.0,0.0,0.0,0.0
2022,0.0,0.0,0.0,0.0,0.0


## 1b. Build consolidated targets

Overwrites `prob_donations` with `donations + bintool_donations` and creates a new `prob_remove` column from the three `remove_*` raw channels. Pass-throughs (liquidations / rtv / warehouse) stay as-is. After this, all 5 targets are real columns on `df`.


In [ ]:
new_exprs = []
for consolidated, sources in _CHANNEL_CONSOLIDATION.items():
    if len(sources) == 1 and sources[0] == consolidated:
        continue  # pass-through, nothing to do
    expr = sum((pl.col(s) for s in sources[1:]), pl.col(sources[0])).alias(consolidated)
    new_exprs.append(expr)

if new_exprs:
    df = df.with_columns(new_exprs)
    print(f'rebuilt {len(new_exprs)} consolidated target columns: {[e.meta.output_name() for e in new_exprs]}')

# Show the new consolidated means alongside prob_recovered so we can eyeball the split.
df.select(
    [pl.col('prob_recovered').mean().alias('mean_prob_recovered')]
    + [pl.col(c).mean().alias(f'mean_{c}') for c in RECOVERY_CHANNELS]
)


## 2. Train & save one model per channel

Per channel:
1. Filter to `prob_recovered > 0` and compute `share = prob_k / prob_recovered`.
2. Join the site x GL baseline share (mean / std / count) from training rows.
3. Optuna tune (`n_trials = N_TRIALS`) in logit space with the custom `prob_mae` metric. If `TUNE = False`, grab the cached params from `_DEFAULT_CHANNEL_PARAMS` instead.
4. Refit on the best params with early stopping. Report MAE / RMSE / R² on the 2025 test split (probability space).
5. Attach the baseline + metadata to the booster and upload to S3.


In [7]:
results = {}
prefix = SHARE_MODELS_S3_PREFIX.rstrip('/')

for i, channel in enumerate(RECOVERY_CHANNELS, start=1):
    print(f'\n[{i}/{len(RECOVERY_CHANNELS)}] {channel}')

    X_train, X_test, y_train, y_test, site_gl_baseline, aug_features = _build_channel_splits(
        df, target_col=channel, train_years=TRAIN_YEARS, test_years=TEST_YEARS, feature_cols=feature_cols,
    )
    print(f'  train_nz={len(X_train):,}  test_nz={len(X_test):,}  features={len(aug_features)}')

    if TUNE:
        best_params = _tune_with_optuna(X_train, X_test, y_train, y_test, n_trials=N_TRIALS, channel=channel)
    elif channel in _DEFAULT_CHANNEL_PARAMS:
        best_params = _DEFAULT_CHANNEL_PARAMS[channel]
        tag = 'warm-start (re-tune recommended)' if channel in {'prob_donations', 'prob_remove'} else 'cached pass-through params'
        print(f'  using {tag}: max_depth={best_params["max_depth"]}, lr={best_params["learning_rate"]:.4f}')
    else:
        raise KeyError(f'no cached params for {channel} and TUNE=False; flip TUNE=True to tune from scratch')

    model, metrics = _train_final_model(X_train, X_test, y_train, y_test, best_params)
    print(f'  best_iter={model.best_iteration}  MAE={metrics["mae"]:.4f}  RMSE={metrics["rmse"]:.4f}  R2={metrics["r2"]:.4f}')

    # Attach baseline + metadata so a single joblib artifact is self-contained
    model.site_gl_baseline_ = site_gl_baseline
    model.aug_features_ = aug_features
    model.channel_ = channel
    model.metrics_ = metrics

    # Save to S3
    s3_key = f'{prefix}/{channel}_share.joblib'
    _save_model_to_s3(model, S3_BUCKET, s3_key)

    results[channel] = {
        'model': model,
        'metrics': metrics,
        'best_params': best_params,
        'X_test': X_test,
        'y_test': y_test,
        'aug_features': aug_features,
    }

print(f'\nDone. All {len(RECOVERY_CHANNELS)} channel models trained and saved to S3.')



[1/4] prob_donations
  train_nz=2,444,105  test_nz=936,134  features=241
  using tuned params: max_depth=8, lr=0.0102


2026-05-17 23:27:40,616 INFO Found credentials from IAM Role: BaseNotebookInstanceEc2InstanceRole


  best_iter=144  MAE=0.0823  RMSE=0.1985  R2=-0.1850
  saved s3://msds-26.2-data/model/recovery_channel_share_softmax/prob_donations_share.joblib

[2/4] prob_liquidations
  train_nz=2,444,105  test_nz=936,134  features=241
  using tuned params: max_depth=8, lr=0.1015
  best_iter=16  MAE=0.1492  RMSE=0.2726  R2=-0.0424
  saved s3://msds-26.2-data/model/recovery_channel_share_softmax/prob_liquidations_share.joblib

[3/4] prob_return_to_vendor
  train_nz=2,444,105  test_nz=936,134  features=241
  using tuned params: max_depth=8, lr=0.1944
  best_iter=13  MAE=0.1811  RMSE=0.3096  R2=0.2397
  saved s3://msds-26.2-data/model/recovery_channel_share_softmax/prob_return_to_vendor_share.joblib

[4/4] prob_warehouse_deals_and_gr
  train_nz=2,444,105  test_nz=936,134  features=241
  using tuned params: max_depth=6, lr=0.1060
  best_iter=21  MAE=0.0722  RMSE=0.1795  R2=0.4771
  saved s3://msds-26.2-data/model/recovery_channel_share_softmax/prob_warehouse_deals_and_gr_share.joblib

Done. All 4 chann

## 3. Per-channel metrics

In [8]:
rows = []
for ch, r in results.items():
    rows.append({
        'channel': ch,
        'test_n': len(r['y_test']),
        'test_mean_share': float(np.mean(r['y_test'])),
        'mae': r['metrics']['mae'],
        'rmse': r['metrics']['rmse'],
        'r2': r['metrics']['r2'],
        'best_iter': int(r['model'].best_iteration),
    })

metrics_df = pd.DataFrame(rows).sort_values('test_mean_share', ascending=False).reset_index(drop=True)
metrics_df

,channel,test_n,test_mean_share,mae,rmse,r2,best_iter
0,prob_return_to_vendor,936134,0.3389,0.1811,0.3096,0.2397,13
1,prob_liquidations,936134,0.1751,0.1492,0.2726,-0.0424,16
2,prob_warehouse_deals_and_gr,936134,0.1276,0.0722,0.1795,0.4771,21
3,prob_donations,936134,0.0828,0.0823,0.1985,-0.1850,144


## 4. Tuned parameters

Per-channel hyperparameters Optuna landed on, plus a JSON dump next to the notebook so we don't lose them when the kernel dies.


In [ ]:
import json
from pathlib import Path

params_df = pd.DataFrame({ch: r['best_params'] for ch, r in results.items()}).T

tuned = {
    ch: {**r['best_params'],
         'mae': r['metrics']['mae'],
         'rmse': r['metrics']['rmse'],
         'r2': r['metrics']['r2'],
         'best_iter': int(r['model'].best_iteration)}
    for ch, r in results.items()
}
out_path = Path('stage3_tuned_params_5ch.json')
out_path.write_text(json.dumps(tuned, indent=2))
print(f'saved {out_path.resolve()}')

params_df


## 5. Verify S3 artifacts


In [9]:
s3 = boto3.client('s3')
resp = s3.list_objects_v2(Bucket=S3_BUCKET, Prefix=prefix)

print(f's3://{S3_BUCKET}/{prefix}/')
for obj in resp.get('Contents', []):
    size_kb = obj['Size'] / 1024
    print(f"  {obj['Key']}  ({size_kb:.0f} KB)")

s3://msds-26.2-data/model/recovery_channel_share_softmax/
  model/recovery_channel_share_softmax/prob_donations_share.joblib  (7275 KB)
  model/recovery_channel_share_softmax/prob_liquidations_share.joblib  (5231 KB)
  model/recovery_channel_share_softmax/prob_return_to_vendor_share.joblib  (5130 KB)
  model/recovery_channel_share_softmax/prob_warehouse_deals_and_gr_share.joblib  (4399 KB)
